
# Unit 2 Assignment: Building a Mixture of Experts (MoE) Router

### Name: Souriesh PB  
### SRN: PES2UG23CS588  

---

## 🎯 Objective

Build a Smart Customer Support Router using a Mixture of Experts (MoE) architecture.

Experts:
- Technical Expert
- Billing Expert
- General Expert
- Tool Expert (Bonus)

The Router classifies user queries and forwards them to the correct expert.


In [22]:
!pip install groq python-dotenv

In [23]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

# Set your API key in Colab like:
os.environ["GROQ_API_KEY"] = "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"

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("Please set your GROQ_API_KEY in environment variables.")

client = Groq(api_key=GROQ_API_KEY)

print("Groq Client Initialized Successfully ✅")

Groq Client Initialized Successfully ✅


In [29]:

MODEL_NAME = "llama-3.1-8b-instant"

MODEL_CONFIG = {
    "technical": {
        "system_prompt": """
You are a Technical Support Expert.
- Be precise and structured.
- Provide code examples if needed.
- Explain technical errors clearly.
- Focus on debugging and best practices.
"""
    },
    "billing": {
        "system_prompt": """
You are a Billing Support Expert.
- Be empathetic and polite.
- Explain refund policies clearly.
- Provide step-by-step guidance.
- Focus on subscriptions, payments, and charges.
"""
    },
    "general": {
        "system_prompt": """
You are a friendly General Assistant.
- Answer casually and helpfully.
- Handle general conversation.
"""
    },
    "tool": {
        "system_prompt": """
You are a Tool Expert.
If user asks for real-time price of Bitcoin, respond with the fetched data.
Otherwise respond normally.
"""
    }
}

In [30]:

def route_prompt(user_input):
    routing_prompt = f"""
Classify the following user input into one of these categories:
[technical, billing, general, tool]

Return ONLY the category name.

User Input:
{user_input}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are a strict classifier."},
            {"role": "user", "content": routing_prompt}
        ],
        temperature=0
    )

    category = response.choices[0].message.content.strip().lower()
    return category


In [31]:

def get_bitcoin_price():
    # Mock data (for assignment purpose)
    return "$67,845 (Mock Data)"


In [32]:

def process_request(user_input):
    category = route_prompt(user_input)
    print(f"🔎 Routed to: {category}")

    if category == "tool" and "bitcoin" in user_input.lower():
        price = get_bitcoin_price()
        return f"📈 Current Bitcoin Price: {price}"

    if category not in MODEL_CONFIG:
        category = "general"

    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ],
        temperature=0.7
    )

    return response.choices[0].message.content


In [33]:

# Test Cases

print(process_request("My Python script throws IndexError on line 5."))
print(process_request("I was charged twice for my subscription."))
print(process_request("Tell me a fun fact about space."))
print(process_request("What is the current price of Bitcoin?"))


🔎 Routed to: technical
**IndexError Explanation**

An `IndexError` in Python occurs when you try to access an element in a sequence (such as a list, tuple, or string) by an index that is out of range. This means the index you're trying to use is either negative or greater than or equal to the length of the sequence.

**Example Code**

Let's consider a simple example:

```python
my_list = [1, 2, 3]
index = 5
print(my_list[index])
```

In this case, the `IndexError` will be raised because `index` is 5, which is greater than the length of `my_list` (3).

**Debugging Steps**

To fix the `IndexError`, follow these steps:

1. **Check the length of the sequence**: Verify the length of the sequence you're trying to access.
2. **Verify the index value**: Ensure the index value is within the valid range (0 to length - 1).
3. **Check for negative indices**: If you're using a negative index, ensure it's not too large (e.g., -length).

**Example Solution**

In the case of the provided code, let's a